# อธิบายค่าที่ใช้ก่อนสร้าง State Space (3RRS)

โน้ตบุ๊กนี้ตอบคำถามว่า “ค่าจากโจทย์ถูกเอาไปคำนวณอะไรบ้าง” ก่อนเขียนสมการ state space

ลำดับที่จะทำ
1. กำหนดตัวแปรและหน่วย SI
2. คำนวณพารามิเตอร์ของลูกบอล (`Ib`, `m_eq`, `kr`)
3. ประมาณพลวัต actuator จากสเปก servo (`Ta`)
4. สร้าง Jacobian ระดับควบคุม (`J`, `Jinv`)
5. หา inverse kinematics ของขา 2-link และ linearized actuator gain (`k_act`)
6. สรุปว่าค่าไหนเข้า state space โดยตรง

In [ ]:
import math
import numpy as np

np.set_printoptions(precision=6, suppress=True)

# -----------------------
# 1) ข้อมูลจากโจทย์ + สมมติฐาน
# -----------------------
mb = 2.7e-3
ball_diameter = 40e-3
mp = 756.34e-3
Ix = 4.261e-3
Iz = 8.509e-3
L1 = 100e-3
L2 = 148.2e-3

Rp = 0.12
Rb = 0.18
h0 = 0.20
b = 0.003
g = 9.81
psi = np.array([0.0, 2.0 * np.pi / 3.0, 4.0 * np.pi / 3.0])

# -----------------------
# 2) ลูกบอล: inertia + rolling factor
# -----------------------
rb = ball_diameter / 2.0
Ib = (2.0 / 3.0) * mb * rb**2
m_eq = mb + Ib / rb**2
kr = mb / m_eq

# -----------------------
# 3) Servo speed -> first-order time constant
# -----------------------
omega_nl = np.deg2rad(60.0) / 0.19
Ta = 1.0 / omega_nl

print('=== Parameters ===')
print(f'mb = {mb:.6f} kg, rb = {rb:.6f} m')
print(f'mp = {mp:.6f} kg, Ix = {Ix:.6e}, Iz = {Iz:.6e} kg.m^2')
print(f'L1 = {L1:.6f} m, L2 = {L2:.6f} m')
print(f'Rp = {Rp:.3f} m, Rb = {Rb:.3f} m, h0 = {h0:.3f} m')
print()
print('=== Derived from ball ===')
print(f'Ib = {Ib:.6e} kg.m^2')
print(f'm_eq = {m_eq:.6f} kg')
print(f'kr = {kr:.6f} (acceleration coefficient)')
print(f'kr*g = {kr*g:.6f} m/s^2 per rad')
print()
print('=== Derived from servo ===')
print(f'omega_nl = {omega_nl:.6f} rad/s')
print(f'Ta = {Ta:.6f} s')
print(f'b/mb = {b/mb:.6f} 1/s')

## จาคอเบียนระดับควบคุม: ความสัมพันธ์ระหว่างท่าทางแท่นกับความสูงจุดยึด

กำหนดเวกเตอร์
- $p = [\phi, \theta, h]^T$
- $z = [z_1, z_2, z_3]^T$

เมื่อมุมเอียงมีขนาดเล็ก จะได้ความสัมพันธ์

$$
z_i = y_i\phi - x_i\theta + h,
\quad
x_i = R_p\cos\psi_i,
\quad
y_i = R_p\sin\psi_i
$$

จึงเขียนรวมในรูปเมทริกซ์ได้ว่า

$$
z = Jp
$$

และถ้าต้องการแปลงย้อนกลับจากความสูงทั้ง 3 จุดเป็นท่าทางแท่น ใช้

$$
p = J^{-1}z
$$

In [ ]:
x_anchor = Rp * np.cos(psi)
y_anchor = Rp * np.sin(psi)
J = np.vstack((y_anchor, -x_anchor, np.ones(3))).T
Jinv = np.linalg.inv(J)

print('J =')
print(J)
print('\nJinv =')
print(Jinv)

# ตัวอย่างแปลง pose -> anchor heights
phi_ex = np.deg2rad(3.0)
theta_ex = np.deg2rad(-2.0)
p_ex = np.array([phi_ex, theta_ex, h0])
z_ex = J @ p_ex
print('\nExample p = [3 deg, -2 deg, h0]')
print('z =', z_ex)

## อินเวิร์สคิเนเมติกส์ของขา 2 ลิงก์ และการหา $k_{act}$

เหตุผลที่ต้องคำนวณส่วนนี้ก่อนสร้าง state space
- วงควบคุมชั้นนอกมักสั่งที่ระดับแท่น: $\phi, \theta, h$
- แอคชูเอเตอร์จริงรับคำสั่งเป็นมุมเซอร์โว
- จึงต้องมีตัวแปลงจากความสูงจุดยึดไปเป็นมุมขาใกล้จุดทำงาน

หาเกนเชิงเส้นใกล้ $h_0$ ด้วย finite difference:

$$
\frac{dq_1}{dz}
\approx
\frac{q_1(h_0+\Delta z)-q_1(h_0-\Delta z)}{2\Delta z}
$$

และนิยาม

$$
k_{act}=\left(\frac{dq_1}{dz}\right)^{-1}
$$

สรุปคือ $k_{act}$ ใช้เชื่อมคำสั่งระดับแท่นไปสู่คำสั่งระดับมุมเซอร์โวในบริเวณจุดทำงาน

In [ ]:
def rotation_matrix_z(angle):
    c = math.cos(angle)
    s = math.sin(angle)
    return np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])


def base_point(index):
    angle = psi[index]
    return np.array([Rb * math.cos(angle), Rb * math.sin(angle), 0.0])


def platform_anchor_world(phi, theta, h, index):
    x_local = Rp * math.cos(psi[index])
    y_local = Rp * math.sin(psi[index])
    anchor_local = np.array([x_local, y_local, 0.0])

    cx, sx = math.cos(phi), math.sin(phi)
    cy, sy = math.cos(theta), math.sin(theta)
    Rx = np.array([[1.0, 0.0, 0.0], [0.0, cx, -sx], [0.0, sx, cx]])
    Ry = np.array([[cy, 0.0, sy], [0.0, 1.0, 0.0], [-sy, 0.0, cy]])
    R = Ry @ Rx
    return np.array([0.0, 0.0, h]) + R @ anchor_local


def world_to_leg_plane(target_world, index):
    leg_yaw = psi[index] + math.pi
    rel_world = np.asarray(target_world) - base_point(index)
    return rotation_matrix_z(-leg_yaw) @ rel_world


def solve_leg_ik(phi, theta, h, index, elbow_sign=1.0):
    target_world = platform_anchor_world(phi, theta, h, index)
    rel_leg = world_to_leg_plane(target_world, index)
    x_local = rel_leg[0]
    z_local = rel_leg[2]

    cos_q2 = (x_local**2 + z_local**2 - L1**2 - L2**2) / (2.0 * L1 * L2)
    cos_q2 = float(np.clip(cos_q2, -1.0, 1.0))
    q2 = elbow_sign * math.acos(cos_q2)

    k1 = L1 + L2 * math.cos(q2)
    k2 = L2 * math.sin(q2)
    q1 = math.atan2(z_local, x_local) - math.atan2(k2, k1)
    return q1, q2, rel_leg


def actuator_gain_from_numerical_linearization(index=0, dz=1e-4):
    q1_minus, _, _ = solve_leg_ik(0.0, 0.0, h0 - dz, index)
    q1_plus, _, _ = solve_leg_ik(0.0, 0.0, h0 + dz, index)
    dq_dz = (q1_plus - q1_minus) / (2.0 * dz)
    return 1.0 / dq_dz


neutral_results = []
for leg_index in range(3):
    q1, q2, rel = solve_leg_ik(0.0, 0.0, h0, leg_index)
    neutral_results.append((q1, q2, rel))

k_act_values = [actuator_gain_from_numerical_linearization(i) for i in range(3)]
k_act = float(np.mean(k_act_values))

for leg_index, (q1, q2, rel) in enumerate(neutral_results, start=1):
    print(f'Leg {leg_index}: rel={rel}, q1={q1:.6f} rad ({np.rad2deg(q1):.3f} deg), q2={q2:.6f} rad ({np.rad2deg(q2):.3f} deg)')

print('\nk_act per leg (m/rad)=', np.array(k_act_values))
print(f'Average k_act = {k_act:.6f} m/rad')

## สรุปว่าค่าไหนเข้า state space โดยตรง

สมการหลักของวงควบคุมชั้นนอก (ต่อหนึ่งแกน) ที่ใช้บ่อยคือ

$$
\ddot{x} = -\frac{b}{m_b}\dot{x} + k_r g\alpha
$$

$$
\dot{\alpha} = -\frac{1}{T_a}\alpha + \frac{1}{T_a}u
$$

ดังนั้น
- ค่าที่เข้าเมทริกซ์พลวัตโดยตรง: $k_r$, $b/m_b$, $T_a$
- ค่าที่ใช้เป็นสะพานคำสั่งก่อนเข้าแอคชูเอเตอร์จริง: $J$, $J^{-1}$, $k_{act}$
- ค่าที่สำคัญกับโมเดลละเอียดขึ้น (แต่ยังไม่เข้าตรงในแบบย่อ): $m_p$, $I_x$, $I_z$